In [2]:
# importing necessary libraries

import os
import numpy as np
import pandas as pd
import xarray as xr
from shapely.geometry import Polygon, Point
from scipy.special import gamma
import warnings
warnings.filterwarnings('ignore')

from py_wake.site import XRSite
from py_wake.wind_turbines.generic_wind_turbines import GenericWindTurbine
from py_wake.wind_turbines import WindTurbines
from py_wake.literature import Nygaard_2022

In [3]:
# file paths

DATA_DIR = "../data"
ERA5_DIR = "../data/era5"
OUTPUT_DIR = "../data/train"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Target Farm data
TARGET_N_TURBINES = 44
TARGET_ROTOR_DIA_M = 164 
TARGET_HUB_HEIGHT_M = 105
TARGET_RATED_POWER = 8400
TARGET_SPACING_D = 5

# Neighbor farm scenaios
NB_DISTANCES_KM = [5, 10, 15, 20, 25, 30, 50]
NB_DIRECTION_DEG = np.arange(0, 360, 30)
NB_SIZES = [20, 40, 80, 120, 160, 200]
NB_SPACINGS_D = [3, 5, 7, 9, 15]
TI_VALUES = [0.04, 0.06, 0.08, 0.1]

# NEIGHBOUR TURBINE SPECS
NEIGHBOUR_SPECS = [
    {'name': 'V112_3300kW', 'diameter': 112, 'hub_height': 84, 'rated_power': 3300},
    {'name': 'SWT_4000kW', 'diameter': 130, 'hub_height': 89, 'rated_power': 4000},
    {'name': 'V164_8400kW', 'diameter': 164, 'hub_height': 105, 'rated_power': 8400},
]

RANDOM_SEED = 42

n_scenarios = (len(NB_DISTANCES_KM) * len(NB_DIRECTION_DEG) * len(NB_SIZES) 
               * len(NB_SPACINGS_D) * len(TI_VALUES) * len(NEIGHBOUR_SPECS))
print(f"Total scenarios: {n_scenarios}")


Total scenarios: 30240


### Load Wind rose from ERA5

In [4]:
def load_era5_wind_rose(data_dir):
    
    from scipy.stats import weibull_min

    #loading ERA5 data
    ds = xr.open_mfdataset(os.path.join(ERA5_DIR, 'era5_norther_100m_*.nc'))
    u = ds['u100'].values.flatten()
    v = ds['v100'].values.flatten()

    ws = np.sqrt(u**2 + v**2)
    wd = (270 - np.degrees(np.arctan2(v, u))) % 360

    sectors = np.arange(0, 360, 30)
    rows = []

    for sector in sectors:
        lower = (sector - 15) % 360
        upper = (sector + 15) % 360

        if lower > upper:
            mask = (wd >= lower) | (wd < upper)
        else:
            mask = (wd >= lower) & (wd < upper)

        ws_sector = ws[mask]
        ws_sector = ws_sector[ws_sector > 0.5 ] # to remove calm winds or no wind

        freq = mask.sum() / len(ws)

        K, _, A = weibull_min.fit(ws_sector, floc=0)
        mean_ws_sector = A * gamma(1 + 1/K)

        rows.append({
            'direction_deg': sector,
            'frequency': freq,
            'weibull_A': A,
            'weibull_k': K,
            'mean_ws': mean_ws_sector,
        })

    wind_rose_df = pd.DataFrame(rows)
    # Normalise frequencies to sum to 1
    wind_rose_df['frequency'] = (wind_rose_df['frequency'] / wind_rose_df['frequency'].sum()) * 100
    
    return wind_rose_df
    

In [5]:
# Load the wind rose
wind_rose = load_era5_wind_rose(ERA5_DIR)
print(wind_rose.head())

   direction_deg  frequency  weibull_A  weibull_k   mean_ws
0              0   5.603710   8.472460   2.294175  7.505663
1             30   8.185067   8.867088   2.797608  7.895451
2             60   7.124392   8.886848   2.641675  7.897245
3             90   5.603710   8.156268   2.536340  7.239430
4            120   5.322384   7.255386   2.426867  6.433157


### Load Target Farm

In [5]:
import geopandas as gpd

gdf = gpd.read_file(os.path.join(DATA_DIR, 'candidate_farms.gpkg'))
target = gdf[gdf['name'].str.contains('Norther', case=False, na=False)].iloc[0]
target_poly = target.geometry

if gdf.crs.is_geographic:
    gdf_proj = gdf.to_crs(epsg=32361)
    target_proj = gdf_proj[gdf_proj['name'].str.contains('Nother', case=False, na=False).iloc[0]]
    target_poly = target_proj.geometry

target_centroid = target_poly.centroid
target_area_sqkm = target_poly.area/ 1e6

print(f"Target area: {target_area_sqkm} km, target centroid: {target_centroid}")

Target area: 38.41130685089931 km, target centroid: POINT (500911.69264268246 5708425.632954922)


### Placing turbines randomly inside the polygon

In [6]:
def place_turbines(polygon, n_turbines, min_spacing_m, seed, max_expand=3):
    current_poly = polygon
    rng = np.random.default_rng(seed)

    for expand_attempt in range(max_expand+1):
        min_x, min_y, max_x, max_y = polygon.bounds
    
        x_placed, y_placed = [], []
        attempts = 0
        max_attempts = n_turbines * 500
    
        while len(x_placed) < n_turbines and attempts < max_attempts:
            x = rng. uniform(min_x, max_x)
            y = rng. uniform(min_y, max_y)

            if not current_poly.contains(Point(x, y)):
                attempts +=1
                continue
                
            too_close = False
            for xi, yi in zip(x_placed, y_placed):
                if (x-xi)**2 + (y-yi)**2 < min_spacing_m **2:
                    too_close = True
                    break
            if not too_close:
                x_placed.append(x)
                y_placed.append(y)

            attempts += 1

        # check if n_turbines were placed
        if len(x_placed) >= n_turbines:
            return np.array(x_placed), np.array(y_placed), current_poly

        # expand the area by 25% if not enough turbines were placed
        if expand_attempt < max_expand:
            scaling_factor = 1.25
            centroid = current_poly.centroid
            from shapely.affinity import scale
            current_poly = scale(current_poly, xfact=scaling_factor, yfact=scaling_factor)

    #Return the turbines placed
    return np.array(x_placed), np.array(y_placed), current_poly
    

In [39]:
def make_neighbour_polygon(target_centroid_x, target_centroid_y, distance_km, direction_deg,
                           n_turbines, spacing_D, rotor_dia):
    n_per_side = int(np.ceil(np.sqrt(n_turbines)))
    spacing_m = spacing_D * rotor_dia
    side_length = (n_per_side + 1) * spacing_m
    
    # Neighbour centroid
    distance_m = distance_km * 1000
    direction_rad = np.radians(direction_deg)
    nb_x = target_centroid_x + distance_m * np.sin(direction_rad)
    nb_y = target_centroid_y + distance_m * np.cos(direction_rad)

    #creating a square polygon
    poly = Polygon([(nb_x - side_length/2, nb_y - side_length/2),
                    (nb_x + side_length/2, nb_y - side_length/2),
                    (nb_x + side_length/2, nb_y + side_length/2),
                    (nb_x - side_length/2, nb_y + side_length/2)])

    return poly, nb_x, nb_y
    

In [40]:
def get_wind_features( wind_rose_df, deg):
    """
    Extract wind rose features for the direction FROM which the neighbour sits.
    
    Parameters:
        wind_rose_df: DataFrame with columns [direction_deg, frequency, mean_ws]
        bearing_deg: bearing from target to neighbour (degrees, 0=N, clockwise)
    
    Returns:
        dict with freq_from_nb_dir, mean_ws_from_nb_dir, dominant_wd_offset
    """
    wind_sectors = wind_rose_df['direction_deg'].values

    # Finding the nearest sector to the direction degree
    diffs = np.abs( wind_sectors - deg)
    diffs = np.minimum(diffs, 360-diffs)
    idx = np.argmin(diffs)

    # Dominant wind direction
    dominant_wd = wind_sectors[wind_rose_df['frequency'].values.argmax()]

    # how far is the direction given from the dominant wind direction
    offset = deg - dominant_wd
    offset = (offset + 180) % 360 - 180 # normalizing to [-180, 180]
    return {
        'freq_from_nb_dir': float(wind_rose_df['frequency'].iloc[idx]),
        'mean_ws_from_nb_dir': float(wind_rose_df['mean_ws'].iloc[idx]),
        'dominant_wd': float(dominant_wd),
        'dominant_wd_offset': float(offset),
        'dominant_wd_offset_sin': float(np.sin(np.radians(offset))),
        'dominant_wd_offset_cos': float(np.cos(np.radians(offset))),
    }


#### Creating function to generate training data

In [43]:
def generate_training_data(wind_rose_df, target_poly, target_centroid, target_spec,
                           neighbour_specs, nb_distances_km, nb_direction_deg, nb_sizes,
                           nb_spacings_D, ti_values, random_seed = 42):
    target_wt = GenericWindTurbine('Target', target_spec['diameter'], target_spec['hub_height'],
                                   target_spec['rated_power'])

    target_spacing_m = TARGET_SPACING_D * target_spec['diameter']
    x_target, y_target, _ = place_turbines( target_poly, TARGET_N_TURBINES, target_spacing_m, seed=random_seed)

    n_target = len(x_target)
    print(f"{len(x_target)} turbines placed")

    scenarios = []
    for distance in nb_distances_km:
        for deg in nb_direction_deg:
            for size in nb_sizes:
                for spacing in nb_spacings_D:
                    for ti in ti_values:
                        for spec in neighbour_specs:
                            scenarios.append({'distance_km': distance,
                                              'direction_deg': float(deg),
                                              'n_nb_turbines': size,
                                              'nb_spacing_D': spacing,
                                              'ti': ti,
                                              'nb_spec': spec})

    print('Total scenrios: ', len(scenarios))
    print("Running Simulations")

    results = []

    for i, sc in enumerate(scenarios):
        try:
            spec = sc['nb_spec']

            # Create neighbour polygon
            nb_poly, nb_x, nb_y = make_neighbour_polygon(target_centroid.x, target_centroid.y,
                                                         sc['distance_km'], sc['direction_deg'],
                                                         sc['n_nb_turbines'],  sc['nb_spacing_D'], 
                                                         spec['diameter'])

            #place neighbour turbines
            nb_spacing_m = sc['nb_spacing_D'] * spec['diameter']
            x_nb, y_nb, nb_poly_used = place_turbines(nb_poly, sc['n_nb_turbines'],
                                                      nb_spacing_m, seed = random_seed)

            if len(x_nb) < sc['n_nb_turbines'] * 0.8:
                print (f"Skipped scenario {i} because less than 80%of the turbines were placed inside the polygon")
                continue

            nb_wt = GenericWindTurbine('Neighbour', spec['diameter'], spec['hub_height'],
                                       spec['rated_power'])


            # Creating Multi turbine object

            wts = WindTurbines.from_WindTurbine_lst([target_wt, nb_wt])

            types_target = [0] * n_target
            types_nb = [1] * len(x_nb)
            types_all = types_target + types_nb

            # Combines positions
            x_all = np.concatenate([x_target, x_nb])
            y_all = np.concatenate([y_target, y_nb])

            # Create Site object

            weibull_A = wind_rose_df['weibull_A'].values
            weibull_k = wind_rose_df['weibull_k'].values
            sector_freq = wind_rose_df['frequency'].values
            wd_centres = wind_rose_df['direction_deg'].values

            ds = xr.Dataset(data_vars = {'Sector_frequency': ('wd', sector_freq),
                                         'Weibull_A': ('wd', weibull_A),
                                         'Weibull_k': ('wd', weibull_k),
                                         'TI': sc['ti'],
                                        }, coords={'wd': wd_centres})

            site = XRSite(ds)

            # Creating wind farm model
            wfm = Nygaard_2022(site, wts)

            sim_alone = wfm(x_target, y_target, type = types_target)

            aep_alone = float(sim_alone.aep().isel({'wt': slice(0, n_target)}).sum().values)

            sim_with_nb = wfm(x_all, y_all, type = types_all)
            aep_with_nb = float(sim_with_nb.aep().isel({'wt': slice(0, n_target)}).sum().values)

            if aep_alone > 0:
                wake_loss_pct = (1 - aep_with_nb / aep_alone) * 100
                
            else:
                wake_loss_pct = 0.0

            wind_features = get_wind_features(wind_rose_df, sc['direction_deg'])

            target_capacity_mw = n_target * target_spec['rated_power']/1000
            target_area_sqkm = target_poly.area / 1e6
            nb_area_sqkm = nb_poly.area / 1e6
            print(sc)
            print("spacing_D:", sc["nb_spacing_D"])
            print("rotor_diam:", spec["diameter"])
            print("spacing_m:", sc["nb_spacing_D"] * spec["diameter"])
            print("n_turbines:", sc["n_nb_turbines"])

            print(nb_poly)
            print(nb_poly.area / 1e6)
            print(nb_poly.is_valid)

            print(target_poly.crs if hasattr(target_poly, "crs") else "polygon has no CRS")
            print(target_centroid.x, target_centroid.y)
            print(target_poly.area)
            
            print(f"target area: {target_area_sqkm}, neighor area: {nb_area_sqkm}")
                
            nb_capacity_mw = len(x_nb) * spec['rated_power'] / 1000

            row = {
                # Target features
                'target_n_turbines': n_target,
                'target_rotor_diam_m': target_spec['diameter'],
                'target_rated_power_kw': target_spec['rated_power'],
                'target_capacity_mw': target_capacity_mw,
                'target_area_sqkm': target_area_sqkm,
                'target_capacity_density': target_capacity_mw / target_area_sqkm,
                
                # Neighbour features
                'nb_n_turbines': len(x_nb),
                'nb_rotor_diam_m': spec['diameter'],
                'nb_hub_height_m': spec['hub_height'],
                'nb_rated_power_kw': spec['rated_power'],
                'nb_capacity_mw': nb_capacity_mw,
                'nb_area_sqkm': nb_area_sqkm,
                'nb_capacity_density': nb_capacity_mw / nb_area_sqkm,
                'nb_spacing_D': sc['nb_spacing_D'],
                
                # Geometry features
                'distance_km': sc['distance_km'],
                'direction_deg': sc['direction_deg'],
                'direction_sin': np.sin(np.radians(sc['direction_deg'])),
                'direction_cos': np.cos(np.radians(sc['direction_deg'])),
                
                # Wind rose features
                'freq_from_nb_dir': wind_features['freq_from_nb_dir'],
                'mean_ws_from_nb_dir': wind_features['mean_ws_from_nb_dir'],
                'dominant_wd': wind_features['dominant_wd'],
                'dominant_wd_offset': wind_features['dominant_wd_offset'],
                'dominant_wd_offset_sin': wind_features['dominant_wd_offset_sin'],
                'dominant_wd_offset_cos': wind_features['dominant_wd_offset_cos'],
                
                # Atmospheric
                'ti': sc['ti'],
                
                # TARGET
                'wake_loss_pct': wake_loss_pct
            }
            
            results.append(row)

        except Exception as e:
            raise e

        if (i + 1) % 100 == 0:
            print(f"  [{i+1}/{len(scenarios)}] completed, rows: {len(results)}")

    df = pd.DataFrame(results)
    #df['wake_loss_pct'] =  df['wake_loss_pct'].clip(lower=0)

    df.to_csv(os.path.join(OUTPUT_DIR, 'training_data.csv'))

    return df

            

In [44]:
# Define target spec
target_spec = {
    'diameter': TARGET_ROTOR_DIA_M,
    'hub_height': TARGET_HUB_HEIGHT_M,
    'rated_power': TARGET_RATED_POWER,
}

# Generate training data
df = generate_training_data(
    wind_rose_df=wind_rose,
    target_poly=target_poly,
    target_centroid=target_centroid,
    target_spec=target_spec,
    neighbour_specs=NEIGHBOUR_SPECS,
    nb_distances_km=NB_DISTANCES_KM,
    nb_direction_deg=NB_DIRECTION_DEG,
    nb_sizes=NB_SIZES,
    nb_spacings_D=NB_SPACINGS_D,
    ti_values=TI_VALUES,
    random_seed=RANDOM_SEED)

print(df.head())

44 turbines placed
Total scenrios:  30240
Running Simulations
{'distance_km': 5, 'direction_deg': 0.0, 'n_nb_turbines': 20, 'nb_spacing_D': 3, 'ti': 0.04, 'nb_spec': {'name': 'V112_3300kW', 'diameter': 112, 'hub_height': 84, 'rated_power': 3300}}
spacing_D: 3
rotor_diam: 112
spacing_m: 336
n_turbines: 20
POLYGON ((499903.69264268246 5712417.632954922, 501919.69264268246 5712417.632954922, 501919.69264268246 5714433.632954922, 499903.69264268246 5714433.632954922, 499903.69264268246 5712417.632954922))
4.064256
True
polygon has no CRS
500911.69264268246 5708425.632954922
38411306.85089931
target area: 38.41130685089931, neighor area: 4.064256


In [ ]:
# Save final output
df.to_csv(os.path.join(OUTPUT_DIR, 'training_data_norther.csv'), index=False)